In [4]:
# Define source path for journal results
SOURCE_PATH = "../../save_and_results/old/journal/"

In [5]:
import sys
sys.path.append('..')

import pickle
import numpy as np
import pandas as pd
from utils.display_tools import load_best_forecasts

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
lot = pickle.load(open("../../save_and_results/cache_lot.p", "rb"))
data = pickle.load(open("../../save_and_results/cache_endog.p", "rb"))
exogs = pickle.load(open("../../save_and_results/cache_exogs.p", "rb"))


chronos = pd.read_csv(SOURCE_PATH + "chronos/Moehne_desc1_chronos_zero_shot_chronos.csv", index_col=0)
tfm = pd.read_csv(SOURCE_PATH + "tfm/Moehne_desc1_tfm_zero_shot_chronos.csv", index_col=0)

d = data["Moehne"]["desc1"]
y_ = d.loc[d.index.year == 2020]

### Table for the PS points

In [8]:
forec = {}
model_stack = {}
for x in [str(n) for n in range(0,4)]:
    model_spec, forecasts = load_best_forecasts(SOURCE_PATH +  x, direction="desc1")
    model_stack[x] = model_spec

    if (x == "1")  or (x == "2"):
        counter = [x for x in forecasts.keys() if "sarimax" not in x]
        counter = [x for x in counter if "var" not in x]
    else:
        counter = forecasts.keys()
    forec[x] = {x: np.abs(forecasts[x] - y_.values).mean() for x in counter}
    forec[x] = pd.concat(forec[x], axis=1).round(3)



chronos = pd.read_csv(SOURCE_PATH + "chronos/Moehne_desc1_chronos_zero_shot_chronos.csv", index_col=0)
tfm = pd.read_csv(SOURCE_PATH + "tfm/Moehne_desc1_tfm_zero_shot_chronos.csv", index_col=0)
# append foundational models
forec["3"]["chronos"] = np.abs(y_ - chronos.values[:-1]).mean().round(3)
forec["3"]["tfm"] = np.abs(y_ - tfm.values[:-1]).mean().round(3)

forec["0"]["chronos"] = np.abs(y_ - chronos.values[:-1]).mean().round(3)
forec["0"]["tfm"] = np.abs(y_ - tfm.values[:-1]).mean().round(3)

In [9]:
order = ["linear", "sarimax", "var", "forest", "ada", "tfm", "chronos"]
order1 = ["linear", "forest", "ada"]
order2 = ["linear", "sarimax", "forest", "ada", "tfm", "chronos"]

In [10]:
out = pd.concat([forec["1"][order1], forec["2"][order1], forec["3"][order2], forec["0"][order]],axis=1).T.round(2)

In [11]:
out.index = [x if x != "sarimax" else "ari" for x in out.index.str[:3]]

In [12]:
print(out.to_latex())

\begin{tabular}{lrrrrrrrr}
\toprule
 & 10905 & 10961 & 10982 & 11038 & 11161 & 11176 & 11208 & 11243 \\
\midrule
lin & 1.520000 & 1.020000 & 1.180000 & 3.850000 & 2.050000 & 1.210000 & 1.560000 & 3.160000 \\
for & 3.410000 & 3.030000 & 3.710000 & 5.010000 & 3.030000 & 3.080000 & 2.760000 & 4.180000 \\
ada & 3.480000 & 3.060000 & 3.620000 & 4.840000 & 2.830000 & 3.070000 & 2.530000 & 3.930000 \\
lin & 1.560000 & 0.980000 & 1.120000 & 3.640000 & 2.090000 & 1.000000 & 1.420000 & 3.260000 \\
for & 1.720000 & 1.390000 & 1.240000 & 4.180000 & 2.230000 & 1.270000 & 1.490000 & 3.270000 \\
ada & 1.570000 & 1.810000 & 1.340000 & 4.660000 & 2.190000 & 1.180000 & 1.570000 & 3.510000 \\
lin & 1.500000 & 0.810000 & 1.060000 & 3.720000 & 2.060000 & 1.010000 & 1.400000 & 3.250000 \\
sar & 1.340000 & 0.940000 & 1.060000 & 3.730000 & 2.050000 & 1.020000 & 1.390000 & 3.260000 \\
for & 1.640000 & 1.020000 & 1.500000 & 4.090000 & 2.220000 & 1.170000 & 1.760000 & 3.270000 \\
ada & 1.400000 & 0.840000 & 0.97

In [13]:
pd.concat([forec[x].T for x in ["1", "2", "3", "0"]])

,10905,10961,10982,11038,11161,11176,11208,11243
forest,3.411,3.031,3.713,5.006,3.026,3.079,2.757,4.182
linear,1.523,1.021,1.179,3.847,2.047,1.206,1.563,3.156
ada,3.475,3.064,3.617,4.839,2.833,3.070,2.529,3.933
forest,1.718,1.391,1.237,4.179,2.234,1.271,1.486,3.266
linear,1.565,0.983,1.120,3.639,2.090,0.995,1.421,3.256
ada,1.573,1.812,1.336,4.664,2.191,1.180,1.567,3.512
forest,1.644,1.020,1.500,4.087,2.221,1.171,1.765,3.267
linear,1.499,0.808,1.065,3.722,2.063,1.013,1.401,3.249
ada,1.398,0.841,0.969,3.676,2.103,1.120,1.647,3.113
sarimax,1.342,0.945,1.063,3.733,2.048,1.016,1.394,3.261


In [14]:
final_naming = ["Full search", "Baseline", "Full Exogenous","Univariate"]
final = []
for n,x in enumerate(forec.keys()):
    print(x)
    a = forec[x].min(axis=1).round(2)
    naming = [forec[x].T.sort_values(y).index[0] for y in a.index]
    naming = [x[:3] if x != "sarimax" else "ari" for x in naming ]
    a = "\makecell{" + a.astype(str) + " // (" + naming +  ")}"

    final.append(a)
final = pd.concat(final, axis=1)
final.columns = final_naming

0
1
2
3


<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_98738/3587143499.py:8: SyntaxWarning: invalid escape sequence '\m'
  a = "\makecell{" + a.astype(str) + " // (" + naming +  ")}"


In [15]:
for x in final.columns:
    final[x] = final[x].str.replace("sar", "ari")

In [16]:
final = final[["Baseline","Full Exogenous", "Univariate", "Full search"]]
final.columns = ["Baseline", "\makecell{Baseline + \\ Exogenous.}", "Univariate", "\makecell{Full model \\ search space}"]

<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_98738/1023676759.py:2: SyntaxWarning: invalid escape sequence '\m'
  final.columns = ["Baseline", "\makecell{Baseline + \\ Exogenous.}", "Univariate", "\makecell{Full model \\ search space}"]
/tmp/ipykernel_98738/1023676759.py:2: SyntaxWarning: invalid escape sequence '\m'
  final.columns = ["Baseline", "\makecell{Baseline + \\ Exogenous.}", "Univariate", "\makecell{Full model \\ search space}"]


In [17]:
final.T

,10905,10961,10982,11038,11161,11176,11208,11243
Baseline,\makecell{1.52 // (lin)},\makecell{1.02 // (lin)},\makecell{1.18 // (lin)},\makecell{3.85 // (lin)},\makecell{2.05 // (lin)},\makecell{1.21 // (lin)},\makecell{1.56 // (lin)},\makecell{3.16 // (lin)}
\makecell{Baseline + \ Exogenous.},\makecell{1.56 // (lin)},\makecell{0.98 // (lin)},\makecell{1.12 // (lin)},\makecell{3.64 // (lin)},\makecell{2.09 // (lin)},\makecell{1.0 // (lin)},\makecell{1.42 // (lin)},\makecell{3.26 // (lin)}
Univariate,\makecell{1.34 // (ari)},\makecell{0.81 // (lin)},\makecell{0.97 // (ada)},\makecell{3.68 // (ada)},\makecell{2.05 // (ari)},\makecell{1.01 // (lin)},\makecell{1.39 // (ari)},\makecell{3.11 // (ada)}
\makecell{Full model \ search space},\makecell{1.5 // (tfm)},\makecell{0.87 // (chr)},\makecell{1.08 // (lin)},\makecell{3.59 // (lin)},\makecell{2.04 // (for)},\makecell{1.06 // (var)},\makecell{1.42 // (lin)},\makecell{3.21 // (ada)}


In [18]:
print(final.T.to_latex())

\begin{tabular}{lllllllll}
\toprule
 & 10905 & 10961 & 10982 & 11038 & 11161 & 11176 & 11208 & 11243 \\
\midrule
Baseline & \makecell{1.52 // (lin)} & \makecell{1.02 // (lin)} & \makecell{1.18 // (lin)} & \makecell{3.85 // (lin)} & \makecell{2.05 // (lin)} & \makecell{1.21 // (lin)} & \makecell{1.56 // (lin)} & \makecell{3.16 // (lin)} \\
\makecell{Baseline + \ Exogenous.} & \makecell{1.56 // (lin)} & \makecell{0.98 // (lin)} & \makecell{1.12 // (lin)} & \makecell{3.64 // (lin)} & \makecell{2.09 // (lin)} & \makecell{1.0 // (lin)} & \makecell{1.42 // (lin)} & \makecell{3.26 // (lin)} \\
Univariate & \makecell{1.34 // (ari)} & \makecell{0.81 // (lin)} & \makecell{0.97 // (ada)} & \makecell{3.68 // (ada)} & \makecell{2.05 // (ari)} & \makecell{1.01 // (lin)} & \makecell{1.39 // (ari)} & \makecell{3.11 // (ada)} \\
\makecell{Full model \ search space} & \makecell{1.5 // (tfm)} & \makecell{0.87 // (chr)} & \makecell{1.08 // (lin)} & \makecell{3.59 // (lin)} & \makecell{2.04 // (for)} & \ma